In [ ]:
import torch
from lightning.pytorch import Trainer
from lightning.pytorch.callbacks import LearningRateMonitor
from lightning.pytorch.loggers import WandbLogger
# from lightning.pytorch.plugins.environments.ddp_environment import DDPPlugin


from solo.methods import BarlowTwins  # imports the method class
from solo.utils.checkpointer import Checkpointer

import lightning.pytorch as L
# DDPStrategy is used instead of DDPPlugin in the latest versions of Lightning
from lightning.pytorch.strategies import DDPStrategy

# Usage
strategy = DDPStrategy(find_unused_parameters=True)
trainer = L.Trainer(strategy="auto", devices=1, accelerator="gpu")


# some data utilities
# we need one dataloader to train an online linear classifier
# (don't worry, the rest of the model has no idea of this classifier, so it doesn't use label info)
from solo.data.classification_dataloader import prepare_data as prepare_data_classification

# and some utilities to perform data loading for the method itself, including augmentation pipelines
from solo.data.pretrain_dataloader import (
    prepare_dataloader,
    prepare_datasets,
    prepare_n_crop_transform,
    build_transform_pipeline, #            prepare_transform,
)

# no use of the below cell

In [ ]:
# common parameters for all methods
# The BarlowTwins class expects an omegaconf.DictConfig with a specific nested structure
from solo.methods import BarlowTwins
import omegaconf

# Configuration must follow the nested structure expected by BaseMethod and BarlowTwins
cfg_dict = {
    "method": "barlow_twins",
    "backbone": {
        "name": "resnet18",
        "kwargs": {
            "zero_init_residual": True,
        }
    },
    "data": {
        "dataset": "cifar10",
        "num_classes": 10,
        "num_large_crops": 2,   # <-- MOVED HERE
        "num_small_crops": 0,   # <-- MOVED HERE
    },
    "max_epochs": 2,
    "accumulate_grad_batches": 1,
    "optimizer": {
        "name": "sgd",
        "batch_size": 256,
        "lr": 0.01,
        "weight_decay": 0.00001,
        "classifier_lr": 0.5,
        "kwargs": {"momentum": 0.9},
        "exclude_bias_n_norm_wd": False,  # <-- RENAMED from exclude_bias_n_norm
    },
    "scheduler": {
        "name": "warmup_cosine",
        "min_lr": 0.0,
        "warmup_start_lr": 0.0,
        "warmup_epochs": 10,
        "lr_decay_steps": None,
        "interval": "step",
    },
    # Barlow Twins specific parameters
    "method_kwargs": {
        "proj_hidden_dim": 2048,
        "proj_output_dim": 2048,
        "lamb": 5e-3,
        "scale_loss": 0.025,
    },
    "name": "barlow-cifar10",
}

# Create the omegaconf DictConfig
cfg = omegaconf.OmegaConf.create(cfg_dict)

# Now create the model with the proper config
model = BarlowTwins(cfg)
print(f"Model created successfully!")
print(f"Backbone: {model.backbone_name}")
print(f"Features dim: {model.features_dim}")

# run this cell

In [ ]:
# common parameters for all methods
# The BarlowTwins class expects an omegaconf.DictConfig with a specific nested structure
from solo.methods import BarlowTwins
import omegaconf

# Configuration must follow the nested structure expected by BaseMethod and BarlowTwins
cfg_dict = {
    "method": "barlow_twins",
    "backbone": {
        "name": "resnet18",
        "kwargs": {
            "zero_init_residual": True,
        }
    },
    "data": {
        "dataset": "cifar10",
        "num_classes": 10,
        "num_large_crops": 2,
        "num_small_crops": 0,
    },
    "max_epochs": 2,
    "accumulate_grad_batches": 1,
    "optimizer": {
        "name": "sgd",
        "batch_size": 256,
        "lr": 0.01,
        "weight_decay": 0.00001,
        "classifier_lr": 0.5,
        "kwargs": {"momentum": 0.9},
        "exclude_bias_n_norm_wd": False,
    },
    "scheduler": {
        "name": "warmup_cosine",
        "min_lr": 0.0,
        "warmup_start_lr": 0.0,
        "warmup_epochs": 10,
        "lr_decay_steps": None,
        "interval": "epoch",
    },
    "method_kwargs": {
        "proj_hidden_dim": 2048,
        "proj_output_dim": 2048,
        "lamb": 5e-3,
        "scale_loss": 0.025,
    },
    "name": "barlow-cifar10",
}

# Create the omegaconf DictConfig
cfg = omegaconf.OmegaConf.create(cfg_dict)

# Now create the model with the proper config
model = BarlowTwins(cfg)
print(f"Model created successfully!")
print(f"Backbone: {model.backbone_name}")
print(f"Features dim: {model.features_dim}")

# ========== DATASET CONFIGURATION ==========
# Augmentation config for CIFAR-10
aug_cfg = omegaconf.OmegaConf.create({
    "crop_size": 32,
    "rrc": {"enabled": True, "crop_min_scale": 0.08, "crop_max_scale": 1.0},
    "color_jitter": {"prob": 0.8, "brightness": 0.4, "contrast": 0.4, "saturation": 0.2, "hue": 0.1},
    "grayscale": {"prob": 0.2},
    "gaussian_blur": {"prob": 0.0},
    "solarization": {"prob": 0.0},
    "equalization": {"prob": 0.0},
    "horizontal_flip": {"prob": 0.5},
})

# Build transform and create N-crop transform (2 large crops)
transform = build_transform_pipeline("cifar10", aug_cfg)
n_crop_transform = prepare_n_crop_transform([transform, transform], num_crops_per_aug=[1, 1])

# Prepare dataset and dataloader
train_dataset = prepare_datasets(dataset="cifar10", transform=n_crop_transform, train_data_path="./data", download=True)
train_loader = prepare_dataloader(train_dataset, batch_size=cfg_dict["optimizer"]["batch_size"], num_workers=4)

print(f"Dataset size: {len(train_dataset)}")
print(f"Number of batches: {len(train_loader)}")

In [ ]:
# we first prepare our single transformation pipeline
transform_cfg = omegaconf.OmegaConf.create({
    "crop_size": 32,  # CIFAR-10 is 32x32
    "rrc": {
        "enabled": True,
        "crop_min_scale": 0.08,
        "crop_max_scale": 1.0,
    },
    "color_jitter": {
        "prob": 0.8,
        "brightness": 0.4,
        "contrast": 0.4,
        "saturation": 0.2,
        "hue": 0.1,
    },
    "grayscale": {"prob": 0.2},
    "gaussian_blur": {"prob": 0.0},
    "solarization": {"prob": 0.0},
    "equalization": {"prob": 0.0},
    "horizontal_flip": {"prob": 0.5},
})
transform = [build_transform_pipeline("cifar10", transform_cfg)]

# then, we wrap the pipeline using this utility function
# to make it produce an arbitrary number of crops
transform = prepare_n_crop_transform(transform, num_crops_per_aug=[2])

# finally, we produce the Dataset/Dataloader classes
train_dataset = prepare_datasets(
    "cifar10",
    transform,
    train_data_path="./data",  # Changed from data_dir
    no_labels=False,
    download=True,
)

# validation dataloader for online evaluation
train_loader_cls, val_loader = prepare_data_classification(
    "cifar10",
    train_data_path="./data",
    val_data_path="./data",
    batch_size=cfg_dict["optimizer"]["batch_size"],
    num_workers=4,
    download=True,
)


# train_loader = prepare_dataloader(
#     train_dataset, 
#     batch_size=cfg_dict["optimizer"]["batch_size"],  # Use cfg_dict instead of base_kwargs
#     num_workers=4
# )

# # validation dataloader for online evaluation
# _, val_loader = prepare_data_classification(
#     "cifar10",
#     data_dir="./data",
#     train_dir=None,
#     val_dir=None,
#     batch_size=cfg_dict["optimizer"]["batch_size"],
#     num_workers=4,
# )

In [ ]:
wandb_logger = WandbLogger(
    name="barlow-cifar10",  # name of the experiment
    project="self-supervised",  # name of the wandb project
    entity=None,
    offline=False,
)
wandb_logger.watch(model, log="gradients", log_freq=100)

In [ ]:
# ========== RECREATE DATALOADERS ==========
transform_cfg = omegaconf.OmegaConf.create({
    "crop_size": 32,
    "rrc": {"enabled": True, "crop_min_scale": 0.08, "crop_max_scale": 1.0},
    "color_jitter": {"prob": 0.8, "brightness": 0.4, "contrast": 0.4, "saturation": 0.2, "hue": 0.1},
    "grayscale": {"prob": 0.2},
    "gaussian_blur": {"prob": 0.0},
    "solarization": {"prob": 0.0},
    "equalization": {"prob": 0.0},
    "horizontal_flip": {"prob": 0.5},
})
transform = [build_transform_pipeline("cifar10", transform_cfg)]
transform = prepare_n_crop_transform(transform, num_crops_per_aug=[2])

# NOTE: num_workers=0 to avoid Windows multiprocessing pickle error
train_dataset = prepare_datasets("cifar10", transform, train_data_path="./data", no_labels=False, download=True)
train_loader = prepare_dataloader(train_dataset, batch_size=cfg_dict["optimizer"]["batch_size"], num_workers=0)
_, val_loader = prepare_data_classification("cifar10", train_data_path="./data", val_data_path="./data", 
                                             batch_size=cfg_dict["optimizer"]["batch_size"], num_workers=0, download=True)

# ========== RECREATE MODEL ==========
cfg = omegaconf.OmegaConf.create(cfg_dict)
model = BarlowTwins(cfg)

# ========== RECREATE CALLBACKS ==========
callbacks = []
lr_monitor = LearningRateMonitor(logging_interval="epoch")
callbacks.append(lr_monitor)
ckpt = Checkpointer(cfg, logdir="checkpoints/barlow", frequency=1)
callbacks.append(ckpt)

# ========== RECREATE TRAINER ==========
trainer = Trainer(
    max_epochs=cfg_dict["max_epochs"],
    logger=wandb_logger,
    callbacks=callbacks,
    strategy="auto",
    accelerator="gpu",
    devices=1,
    accumulate_grad_batches=cfg_dict["accumulate_grad_batches"],
)

# ========== START TRAINING ==========
trainer.fit(model, train_loader, val_loader)

# Load Checkpoints

In [ ]:
# ========== EVALUATE CHECKPOINT WITH KNN ==========
from solo.utils.knn import WeightedKNNClassifier
from tqdm import tqdm

# Load checkpoint
ckpt_path = "checkpoints/barlow/rsaowzxh/barlow-cifar10-rsaowzxh-ep=1.ckpt"  # Update this!
model = BarlowTwins.load_from_checkpoint(ckpt_path, strict=False, cfg=cfg)
model.eval()
model.cuda()

# Extract features
@torch.no_grad()
def extract_features(loader, model):
    features, labels = [], []
    for im, lab in tqdm(loader):
        im = im.cuda()
        out = model(im)
        features.append(out["feats"].cpu())
        labels.append(lab)
    return torch.cat(features), torch.cat(labels)

train_feats, train_labels = extract_features(train_loader_cls, model)
test_feats, test_labels = extract_features(val_loader, model)

# Run KNN
knn = WeightedKNNClassifier(k=20, distance_fx="cosine")
knn(train_features=train_feats.cuda(), train_targets=train_labels.cuda(),
    test_features=test_feats.cuda(), test_targets=test_labels.cuda())
acc1, acc5 = knn.compute()
print(f"KNN Accuracy: Top-1={acc1:.2f}%, Top-5={acc5:.2f}%")